### Importing data set

In [1]:
import pandas as pd
import numpy as np

url="https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"

df=pd.read_csv(url)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### Data cleaning by replacing 0 with NaN

In [3]:
columns_to_fix=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[columns_to_fix]=df[columns_to_fix].replace(0, np.nan)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50,1
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31,0
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


In [4]:
df.isnull().sum()

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

### Train Test Split

In [6]:
from sklearn.model_selection import train_test_split

X=df.drop(columns=['Outcome'], axis=1)
Y=df['Outcome']

X_train, X_test, Y_train, Y_test=train_test_split(X, Y, test_size=0.2, random_state=42)

### Simple Imputer

In [8]:
from sklearn.impute import SimpleImputer

impute=SimpleImputer(strategy='median')

X_train_imputed=impute.fit_transform(X_train)
X_test_imputed=impute.transform(X_test)

### Convert back to data frame

In [9]:
new_X_train=pd.DataFrame(X_train_imputed, columns=X.columns)
new_X_test=pd.DataFrame(X_test_imputed, columns=X.columns)

new_X_train.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,2.0,84.0,72.0,28.5,120.0,32.0,0.304,21.0
1,9.0,112.0,82.0,24.0,120.0,28.2,1.282,50.0
2,1.0,139.0,46.0,19.0,83.0,28.7,0.654,22.0
3,0.0,161.0,50.0,28.5,120.0,21.9,0.254,65.0
4,6.0,134.0,80.0,37.0,370.0,46.2,0.238,46.0


### Applying Z-Score Technique

In [21]:
columns_for_capping=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# applying for loop for all columns

for x in columns_for_capping:

    # find mean
    mean_value=new_X_train[x].mean()

    # find std
    std=new_X_train[x].std()

    # calculate upper limit
    upper_limit= mean_value+ 3*std

    # calculate lower limit
    lower_limit= mean_value - 3*std

    # calculate before max
    before_max=new_X_train[x].max()
    
    # calculate before min
    before_min=new_X_train[x].min()
    
    # Applying capping to train data set
    new_X_train[x]=np.where(
        new_X_train[x]>upper_limit, upper_limit,
        np.where(
            new_X_train[x]<lower_limit, lower_limit,
            new_X_train[x]
            ) 
    )

    # Applying capping to test data set
    new_X_test[x]=np.where(
        new_X_test[x]>upper_limit, upper_limit,
        np.where(
            new_X_test[x]<lower_limit, lower_limit,
            new_X_test[x]
        )
    )


    # calculate after max
    after_max=new_X_train[x].max()
    
    # calculate after min
    after_min=new_X_train[x].min()


    # printing the values for our information
    print(f"""
            Column: {x}
            Mean: {mean_value:.2f}
            STD: {std:.2f}
            Upper_limit: {upper_limit:.2f}
            Lower_limit: {lower_limit:.2f}
            Maximum:
            before: {before_max:.2f} |  after: {after_max:.2f}
            Minimum:
            before: {before_min:.2f} | after: {after_min:.2f}
            """)


            Column: Glucose
            Mean: 121.82
            STD: 30.10
            Upper_limit: 212.13
            Lower_limit: 31.50
            Maximum:
            before: 199.00 |  after: 199.00
            Minimum:
            before: 44.00 | after: 44.00
            

            Column: BloodPressure
            Mean: 72.24
            STD: 11.85
            Upper_limit: 107.80
            Lower_limit: 36.69
            Maximum:
            before: 107.80 |  after: 107.80
            Minimum:
            before: 36.68 | after: 36.69
            

            Column: SkinThickness
            Mean: 28.54
            STD: 8.32
            Upper_limit: 53.50
            Lower_limit: 3.58
            Maximum:
            before: 53.50 |  after: 53.50
            Minimum:
            before: 8.00 | after: 8.00
            

            Column: Insulin
            Mean: 131.80
            STD: 61.63
            Upper_limit: 316.68
            Lower_limit: -53.09
            Maxi

In [23]:
new_X_train.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64